# 04 · Data and Scoring

Score synthetic records or browser exports. Upload the project ZIP when prompted. To import a browser export, set mode to walkthrough or participant and set upload_responses to True in Colab. Locally, place the JSON in private_records.

In [ ]:
import io
import json
import sys
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

root = next((p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "check-before-you-run")
             if (p / "src" / "study.py").is_file()), None)
if root is None:
    from google.colab import files
    uploads = files.upload()
    if len(uploads) != 1:
        raise ValueError("Upload the original project ZIP.")
    with ZipFile(io.BytesIO(next(iter(uploads.values())))) as archive:
        paths = [PurePosixPath(name) for name in archive.namelist()]
        if not paths or any(path.is_absolute() or ".." in path.parts for path in paths):
            raise ValueError("Invalid project archive.")
        archive.extractall(Path.cwd())
    root = Path.cwd() / "check-before-you-run"
    if not (root / "src" / "study.py").is_file():
        raise ValueError("Use the original check-before-you-run.zip.")
root = root.resolve()
sys.path.insert(0, str(root))
from src import study

print("Project ready")

In [ ]:
mode = "synthetic"
response_folder = Path.cwd() / "private_records"
upload_responses = False

if mode == "synthetic":
    records = json.loads((root / "demo" / "synthetic_records.json").read_text())
elif upload_responses:
    from google.colab import files
    records = [json.loads(data) for data in files.upload().values()]
else:
    records = [json.loads(path.read_text()) for path in sorted(response_folder.glob("*.json"))]
rows = study.score_sessions(records, mode)
print(f"{len(records)} {mode} sessions | {len(rows)} scored rounds")

In [ ]:
import csv

out = Path.cwd() / "results"
out.mkdir(exist_ok=True)
destination = out / f"{mode}-scores.csv"
with destination.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print(destination)
print(rows[0])